In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [23]:
train.head()

,row_id,time,x,y,direction,congestion
0,0,1991-04-01 00:00:00,0,0,EB,70
1,1,1991-04-01 00:00:00,0,0,NB,49
2,2,1991-04-01 00:00:00,0,0,SB,24
3,3,1991-04-01 00:00:00,0,1,EB,18
4,4,1991-04-01 00:00:00,0,1,NB,60


In [3]:
train_id = train["row_id"]
test_id = test["row_id"]
train.drop("row_id", axis=1, inplace=True)
test.drop("row_id", axis=1, inplace=True)

In [4]:
train["time"] = pd.to_datetime(train["time"])
train["hour"] = train["time"].dt.hour
train["day"] = train["time"].dt.day
train["month"] = train["time"].dt.month
train["weekday"] = train["time"].dt.weekday

test["time"] = pd.to_datetime(test["time"])
test["hour"] = test["time"].dt.hour
test["day"] = test["time"].dt.day
test["month"] = test["time"].dt.month
test["weekday"] = test["time"].dt.weekday

In [5]:
train.drop("time", axis=1, inplace=True)
test.drop("time", axis=1, inplace=True)

In [6]:
train = pd.concat([train,pd.get_dummies(train["direction"], drop_first=True)], axis=1)
test = pd.concat([test,pd.get_dummies(test["direction"], drop_first=True)], axis=1)

In [7]:
train.drop("direction", axis=1, inplace=True)
test.drop("direction", axis=1, inplace=True)

X = train.drop("congestion", axis=1)
y = train["congestion"]

In [8]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.03, random_state=42)
len(X_train), len(X_test), len(y_train), len(y_test)

(823369, 25466, 823369, 25466)

In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
scaler.fit(X_train)
X_train = scaler.transform(X_train)
X_test = scaler.transform(X_test)
scaler = StandardScaler()
test = scaler.fit_transform(test)

In [ ]:
from xgboost import XGBRegressor

xgbr = XGBRegressor(n_estimators=1500, learning_rate=0.1, max_depth=10)
xgbr.fit(X_train, y_train)

In [14]:
from life_saving_tools.Notification import Notification

n = Notification()
n.play_n_stop(5)

In [15]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

In [16]:
mean_absolute_error(y_test, xgbr.predict(X_test)), mean_absolute_error(y_train, xgbr.predict(X_train))

(5.887336159870292, 5.008519372661817)

## ML Models

### Polynomial Regression

In [28]:
from sklearn.preprocessing import PolynomialFeatures
pr = PolynomialFeatures(degree=2)
X_train_pr = pr.fit_transform(X_train)
X_test_pr = pr.fit_transform(X_test)

In [29]:
X_train_pr.shape

(823369, 105)

In [31]:
from sklearn.linear_model import LinearRegression
lr = LinearRegression()
lr.fit(X_train_pr, y_train)

y_pred = lr.predict(X_test_pr)
rmse = (mean_squared_error(y_test, y_pred))**0.5
mae = mean_absolute_error(y_test, y_pred)

text = f"""Training Complete!
Root Mean Squared Error: {rmse}
Mean Absolute Error: {mae}"""
print(text)
# n.send_whatsapp_text(text)

Training Complete!
Root Mean Squared Error: 13.577212206765623
Mean Absolute Error: 10.706950259659939


### SVR

In [32]:
from sklearn.svm import SVR

svr = SVR(kernel='rbf')
svr.fit(X_train, y_train)

In [ ]:
y_pred = svr.predict(X_test_pr)
rmse = (mean_squared_error(y_test, y_pred))**0.5
mae = mean_absolute_error(y_test, y_pred)

text = f"""Training Complete!
Root Mean Squared Error: {rmse}
Mean Absolute Error: {mae}"""
print(text)

In [ ]:
n.send_whatsapp_text(text)

### Random Forest

In [31]:
from sklearn.ensemble import RandomForestRegressor
rfr = RandomForestRegressor(n_estimators=1000, max_depth=10)
rfr.fit(X_train, y_train)

y_pred = svr.predict(X_test_pr)
rmse = (mean_squared_error(y_test, y_pred))**0.5
mae = mean_absolute_error(y_test, y_pred)

text = f"""Training Complete!
Root Mean Squared Error: {rmse}
Mean Absolute Error: {mae}"""
print(text)


In [ ]:
n.send_whatsapp_text(text)

## Neural Networks

In [10]:
import tensorflow as tf
from tensorflow.keras import layers as tfl
from tensorflow.keras.models import Sequential
from tensorflow.keras import Model

In [11]:
input = tfl.Input(shape=(X_train.shape[1],), name="input")
x = tfl.Dense(100, activation='relu', name="Dense1")(input)
x = tfl.Dense(100, activation='relu',  name="Dense2")(x)
output = tfl.Dense(1, activation="linear", name="output")(x)
model1 = Model(inputs=input, outputs=output, name="model1")

model1.summary()

Model: "model1"
_________________________________________________________________
Layer (type)                 Output Shape              Param #   
input (InputLayer)           [(None, 13)]              0         
_________________________________________________________________
Dense1 (Dense)               (None, 100)               1400      
_________________________________________________________________
Dense2 (Dense)               (None, 100)               10100     
_________________________________________________________________
output (Dense)               (None, 1)                 101       
Total params: 11,601
Trainable params: 11,601
Non-trainable params: 0
_________________________________________________________________


In [ ]:
model1.compile(optimizer='adam', loss='mse', metrics=['mse'])
model1.fit(X_train, y_train, epochs=10, batch_size=256, validation_data=(X_test, y_test))

In [20]:
y_pred = xgbr.predict(test)
assert len(y_pred) == len(test_id)
pd.DataFrame({"row_id": test_id, "congestion": y_pred}).to_csv("submission.csv", index=False)

In [21]:
!kaggle competitions submit -c tabular-playground-series-mar-2022 -f submission.csv -m "XGBoost Final"

Successfully submitted to Tabular Playground Series - Mar 2022



  0%|          | 0.00/40.2k [00:00<?, ?B/s]
 20%|█▉        | 8.00k/40.2k [00:00<00:00, 54.6kB/s]
100%|██████████| 40.2k/40.2k [00:06<00:00, 6.76kB/s]


In [11]:
model = r"C:\Users\harik\Downloads\best_model.h5"
model = tf.keras.models.load_model(model)
model.evaluate(X_test, y_test)

796/796 [==============================] - 9s 10ms/step - loss: 6.2631 - mae: 6.2631: 0s - loss: 6.2476 - ma


[6.263131618499756, 6.263131618499756]

In [18]:
y_pred = model.predict(test)
assert len(y_pred) == len(test_id)
pd.DataFrame({"row_id": test_id, "congestion": tf.squeeze(y_pred)}).to_csv("submission.csv", index=False)

In [19]:
!kaggle competitions submit -c tabular-playground-series-mar-2022 -f submission.csv -m "NN model"

Successfully submitted to Tabular Playground Series - Mar 2022


  0%|          | 0.00/40.0k [00:00<?, ?B/s]
 20%|█▉        | 8.00k/40.0k [00:00<00:00, 51.5kB/s]
100%|██████████| 40.0k/40.0k [00:05<00:00, 7.03kB/s]
